# 02 — Export and correct NEON reflectance

Run the three canonical correction stages explicitly: HDF5 → raw ENVI, correction-parameter JSON, then topo/BRDF-corrected ENVI. Each stage validates and reuses completed files.

In [ ]:
from pathlib import Path
from spectralbridge.pipelines.pipeline import (
    stage_apply_brdf_topo_correction, stage_build_and_write_correction_json,
    stage_export_envi_from_h5,
)

RUN = False
BASE_FOLDER = Path("outputs/neon_notebook")
PRODUCT_CODE = "DP1.30006.001"
FLIGHT_STEM = "NEON_D13_NIWO_DP1_L019-1_20230815_directional_reflectance"
TOPO_FIT_MODE = "scene"

In [ ]:
if RUN:
    raw_img, raw_hdr = stage_export_envi_from_h5(BASE_FOLDER, PRODUCT_CODE, FLIGHT_STEM)
    params_json = stage_build_and_write_correction_json(
        base_folder=BASE_FOLDER, product_code=PRODUCT_CODE, flight_stem=FLIGHT_STEM,
        raw_img_path=raw_img, raw_hdr_path=raw_hdr,
    )
    corrected_img, corrected_hdr = stage_apply_brdf_topo_correction(
        base_folder=BASE_FOLDER, product_code=PRODUCT_CODE, flight_stem=FLIGHT_STEM,
        raw_img_path=raw_img, raw_hdr_path=raw_hdr, correction_json_path=params_json,
        topo_fit_mode=TOPO_FIT_MODE,
    )
    print(raw_img, params_json, corrected_img, sep="\n")
else:
    print("Dry run. Confirm the canonical HDF5 exists, then set RUN = True.")

## Validate
The raw and corrected `.img/.hdr` pairs must share spatial shape and wavelength order. The correction JSON belongs to this flightline and is evidence, not a reusable global coefficient file.